### 2.2 Unicode Encodings(a)

In [ ]:
test_string = "hello 慧影！"
utf8_encoded = test_string.encode("utf-8")
utf16_encoded = test_string.encode("utf-16")
utf32_encoded = test_string.encode("utf-32")
print(type(utf8_encoded))
print(utf8_encoded)
print(type(utf16_encoded))
print(utf16_encoded)
print(type(utf32_encoded))
print(utf32_encoded)
print(len(utf8_encoded), list(utf8_encoded))
print(len(utf16_encoded), list(utf16_encoded))
print(len(utf32_encoded), list(utf32_encoded))
print(utf8_encoded.decode("utf-8"))
print(utf16_encoded.decode("utf-16"))
print(utf32_encoded.decode("utf-32"))

#### (a) 相比于 UTF-16 或 UTF-32，选择在 UTF-8 编码的字节上训练分词器（tokenizer）有哪些优势？
UTF-8编码本质上是将一个码点序列转换成一个长度为256的字节值序列，相比于UTF-16与UTF-32而言，256长度处理起来要容易得多。

### (b)

In [ ]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


decode_utf8_bytes_to_str_wrong("hello 慧影!".encode("utf-8"))

#### 考虑以下（不正确的）函数，其旨在将 UTF-8 字节串解码为 Unicode 字符串。为什么该函数是不正确的？
解码的时是逐个字符进行解码的，对于英文，在UTF-8中是一个字符占一个编码位，所以可以逐个解码；但是对于其他语言，比如中文，可能是一个字符会占多个编码位，所以不能够进行逐个解码，进而会发生报错。

### 2.5 BPE Training on TinyStories

#### 编写一个函数，给定输入文本文件的路径，训练一个（字节级）BPE分词器。
assignment1/bpe.py

### Training on TinyStories
#### 在 TinyStories 数据集上训练一个字节级 BPE 分词器，词汇表最大规模设为 10,000。请确保将 TinyStories 的 <|endoftext|> 特殊标记添加到词汇表中。将生成的词汇表和合并规则序列化并保存到磁盘，以便后续检查。训练过程耗时多久？占用了多少内存？词汇表中最长的标记是什么？这是否合理？

In [1]:
from assignment1.bpe import train_bpe
import time
import resource

input_path = "./data/TinyStoriesV2-GPT4-train.txt"
output_path = "./assignment1/assets/"
vocab_size = 10000
special_tokens = ["<|endoftext|>"]

start = time.perf_counter()
vocab, merges = train_bpe(input_path, vocab_size, special_tokens)
total_time = time.perf_counter() - start

peak_memory = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_memory_gib = peak_memory / (1024 ** 3)

max_token = b""
with open(output_path+"vocab.txt", "w", encoding="utf-8") as file:
    for token_ix, token_bytes in vocab.items():
        max_token = token_bytes if len(token_bytes)>len(max_token) else max_token
        file.write(f"{token_ix}\t{token_bytes!r}\n")
with open(output_path+"merges.txt", "w", encoding="utf-8") as file:
    for left, right in merges:
        file.write(f"{left!r}\t{right!r}\n")

print(f"训练耗时：{total_time:.2f} 秒")
print(f"进程峰值内存：{peak_memory_gib:.2f} GiB")
print(f"词表中最长标记是：{max_token!r} ")

训练耗时：815.94 秒
进程峰值内存：5.55 GiB
词表中最长标记是：b' accomplishment' 


In [1]:
from assignment1.bpe import train_bpe
import time
import resource

input_path = "./data/TinyStoriesV2-GPT4-train.txt"
output_path = "./assignment1/assets/"
vocab_size = 10000
special_tokens = ["<|endoftext|>"]

start = time.perf_counter()
vocab, merges = train_bpe(input_path, vocab_size, special_tokens, 4)
total_time = time.perf_counter() - start

peak_memory = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_memory_gib = peak_memory / (1024**3)

max_token = b""
with open(output_path + "vocab.txt", "w", encoding="utf-8") as file:
    for token_ix, token_bytes in vocab.items():
        max_token = token_bytes if len(token_bytes) > len(max_token) else max_token
        file.write(f"{token_ix}\t{token_bytes!r}\n")
with open(output_path + "merges.txt", "w", encoding="utf-8") as file:
    for left, right in merges:
        file.write(f"{left!r}\t{right!r}\n")

print(f"训练耗时：{total_time:.2f} 秒")
print(f"进程峰值内存：{peak_memory_gib:.2f} GiB")
print(f"词表中最长标记是：{max_token!r} ")

训练耗时：631.47 秒
进程峰值内存：0.16 GiB
词表中最长标记是：b' accomplishment' 


### Training on OpenWebText
####  在 OpenWebText 数据集上训练一个字节级 BPE 分词器，并将词表大小上限设为 32,000。将生成的词表和合并规则序列化并保存至磁盘以供进一步检查。词表中长度最长的 token 是什么？它是否具有实际意义？

In [ ]:
from assignment1.bpe import train_bpe
import time
import resource

input_path = "./data/owt_train.txt"
output_path = "./assignment1/assets/"
vocab_size = 32000
special_tokens = ["<|endoftext|>"]

start = time.perf_counter()
vocab, merges = train_bpe(input_path, vocab_size, special_tokens, 2)
total_time = time.perf_counter() - start

peak_memory = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_memory_gib = peak_memory / (1024**3)

max_token = b""
with open(output_path + "vocab.txt", "w", encoding="utf-8") as file:
    for token_ix, token_bytes in vocab.items():
        max_token = token_bytes if len(token_bytes) > len(max_token) else max_token
        file.write(f"{token_ix}\t{token_bytes!r}\n")
with open(output_path + "merges.txt", "w", encoding="utf-8") as file:
    for left, right in merges:
        file.write(f"{left!r}\t{right!r}\n")

print(f"训练耗时：{total_time:.2f} 秒")
print(f"进程峰值内存：{peak_memory_gib:.2f} GiB")
print(f"词表中最长标记是：{max_token!r} ")

### Implementing the tokenizer

#### 实现一个 Tokenizer 类，该类在给定词表和合并规则列表的情况下，将文本编码为整数 ID，并将整数 ID 解码为文本。您的分词器还应支持用户提供的特殊标记。
assignment1/tokenizer.py

In [29]:
import time
from assignment1.tokenizer import Tokenizer
import resource

# input_path = "./data/TinyStoriesV2-GPT4-train.txt"
input_path = "./data/owt_train.txt"
# output_path = "./assignment1/assets/tiny/"
output_path = "./assignment1/assets/owt/"
vocab_size = 32000
special_tokens = ["<|endoftext|>"]
start = time.perf_counter()
# tokenizer = Tokenizer.train(input_path, vocab_size, special_tokens, 4)
tokenizer = Tokenizer.train(input_path, vocab_size, special_tokens, 2)
total_time1 = time.perf_counter() - start

peak_memory = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_memory_gib = peak_memory / (1024**3)

start = time.perf_counter()
max_token = b""
merges = sorted(tokenizer.merge_ranks, key=tokenizer.merge_ranks.get)
with open(output_path + "vocab.txt", "w", encoding="utf-8") as file:
    for token_ix, token_bytes in tokenizer.vocab.items():
        max_token = token_bytes if len(token_bytes) > len(max_token) else max_token
        file.write(f"{token_ix}\t{token_bytes!r}\n")
with open(output_path + "merges.txt", "w", encoding="utf-8") as file:
    for left, right in merges:
        file.write(f"{left!r}\t{right!r}\n")
total_time2 = time.perf_counter() - start

print(f"训练耗时：{total_time1:.2f} 秒，读写耗时：{total_time2:.6f} 秒")
print(f"进程峰值内存：{peak_memory_gib:.2f} GiB")
print(f"词表中最长标记是：{max_token!r} ")

KeyboardInterrupt: 

#### 从 TinyStories 和 OpenWebText 中各抽取 10 份文档。使用您先前训练的 TinyStories 和 OpenWebText 分词器（词汇表大小分别为 10K 和 32K），将这些抽取的文档编码为整数 ID。每个分词器的压缩比（字节/标记）是多少？并估算分词器吞吐量（例如，以字节/秒为单位）

In [ ]:
import random, time
from assignment1.tokenizer import Tokenizer

input_path = "./data/TinyStoriesV2-GPT4-train.txt"
tiny_path = "./assignment1/assets/tiny/"
owt_path = "./assignment1/assets/owt/"
special_tokens = ["<|endoftext|>"]


def read_documents(input_path, chunk_size=1024 * 1024):
    """逐块读取文件，按文档分隔符产生完整文档。"""
    buffer = ""
    with open(input_path, "r", encoding="utf-8", newline="") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            buffer += chunk
            parts = buffer.split(special_tokens[0])
            # 最后一部分可能是不完整的文档，留待下次读取
            buffer = parts.pop()
            for document in parts:
                if document.strip():
                    yield document
    # 处理文件末尾没有分隔符的文档
    if buffer.strip():
        yield buffer


def sample_documents(input_path, sample_size=10, seed=1536):
    """蓄水池抽样：扫描全部文档，仅保留指定数量的样本。"""
    rng = random.Random(seed)
    samples = []
    for index, document in enumerate(read_documents(input_path)):
        if index < sample_size:
            samples.append(document)
        else:
            selected = rng.randrange(index + 1)
            if selected < sample_size:
                samples[selected] = document
    if len(samples) < sample_size:
        raise ValueError(f"文件中的非空文档不足 {sample_size} 篇。")
    return samples


def evaluate_tokenizer(tokenizer, documents, repeats=3):
    # 统计压缩比
    total_bytes = sum(len(document.encode("utf-8")) for document in documents)
    tokenizer.encode(documents[0])
    durations, expected_tokens = [], None
    for _ in range(repeats):
        total_tokens = 0
        start = time.perf_counter()
        for document in documents:
            total_tokens += len(tokenizer.encode(document))
        durations.append(time.perf_counter() - start)
        if expected_tokens is None:
            expected_tokens = total_tokens
        else:
            assert total_tokens == expected_tokens
    # 合并多次测量，计算平均一次耗时
    average_seconds = sum(durations) / repeats
    throughput = total_bytes / average_seconds
    print(f"文档数量：{len(documents)}")
    print(f"原文总字节数：{total_bytes}")
    print(f"token 总数：{expected_tokens}")
    print(f"压缩比：{total_bytes/expected_tokens:.2f} bytes/token")
    print(f"平均编码耗时：{sum(durations)/repeats:6f} 秒")
    print(f"吞吐量：{throughput:.2f} bytes/s")
    print(f"吞吐量：" f"{throughput / 1024**2:.2f} MiB/s")


tiny_tokenizer = Tokenizer.from_files(
    tiny_path + "vocab.txt", tiny_path + "merges.txt", special_tokens
)
tiny_documents = sample_documents("./data/TinyStoriesV2-GPT4-valid.txt")
evaluate_tokenizer(tiny_tokenizer, tiny_documents)
owt_tokenizer = Tokenizer.from_files(
    owt_path + "vocab.txt", owt_path + "merges.txt", special_tokens
)
owt_documents = sample_documents("./data/owt_valid.txt")
evaluate_tokenizer(owt_tokenizer, owt_documents)

文档数量：10
原文总字节数：6860
token 总数：1679
压缩比：4.09 bytes/token
平均编码耗时：0.003265 秒
吞吐量：2100928.91 bytes/s
吞吐量：2.00 MiB/s


#### 如果使用 TinyStories 分词器对 OpenWebText 样本进行分词，会发生什么？请比较压缩比，并定性描述其结果。

In [ ]:
tiny_tokenizer = Tokenizer.from_files(
    tiny_path + "vocab.txt", tiny_path + "merges.txt", special_tokens
)
owt_documents = sample_documents("./data/owt_valid.txt")
evaluate_tokenizer(tiny_tokenizer, owt_documents)

#### 使用您的 TinyStories 和 OpenWebText 分词器，将相应的训练集和开发集编码为整数标记 ID 序列。我们稍后将使用此序列来训练我们的模型。

In [ ]:
import time
from assignment1.tokenizer import Tokenizer
import resource

# input_path = "./data/TinyStoriesV2-GPT4-valid.txt"
input_path = "./data/owt_valid.txt"
# output_path = "./assignment1/assets/tiny/"
output_path = "./assignment1/assets/owt/"
# vocab_size = 10000
vocab_size = 32000
special_tokens = ["<|endoftext|>"]
start = time.perf_counter()
# tokenizer = Tokenizer.train(input_path, vocab_size, special_tokens, 4)
tokenizer = Tokenizer.train(input_path, vocab_size, special_tokens, 2)
total_time1 = time.perf_counter() - start

peak_memory = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_memory_gib = peak_memory / (1024**3)

start = time.perf_counter()
max_token = b""
merges = sorted(tokenizer.merge_ranks, key=tokenizer.merge_ranks.get)
with open(output_path + "vocab1.txt", "w", encoding="utf-8") as file:
    for token_ix, token_bytes in tokenizer.vocab.items():
        max_token = token_bytes if len(token_bytes) > len(max_token) else max_token
        file.write(f"{token_ix}\t{token_bytes!r}\n")
with open(output_path + "merges1.txt", "w", encoding="utf-8") as file:
    for left, right in merges:
        file.write(f"{left!r}\t{right!r}\n")
total_time2 = time.perf_counter() - start

print(f"训练耗时：{total_time1:.2f} 秒，读写耗时：{total_time2:.6f} 秒")
print(f"进程峰值内存：{peak_memory_gib:.2f} GiB")
print(f"词表中最长标记是：{max_token!r} ")

### 3.2 einstein examples

In [ ]:
import torch
from einops import einsum, rearrange
import einx

images = torch.randn(64, 128, 128, 3)
vec = torch.linspace(start=0, end=1, steps=10)
print(images.shape, vec.shape)
ve = rearrange(vec, "D -> 1 D 1 1 1")
imgs = rearrange(images, "B H W C -> B 1 H W C")
ima_vec = imgs * ve
print(ima_vec.shape)
img_vec = einsum(images, vec, "B H W C, D -> B D H W C")
print(img_vec.shape)

#### 原生pytorch写法

In [ ]:
images = torch.randn(64, 32, 32, 3)
transform = torch.randn(32 * 32, 32 * 32)
# 展平宽和高，然后交换维度
images1 = images.view(-1, images.size(1) * images.size(2), images.size(3))
images2 = images1.transpose(1, 2)
# 应用线形变换
images3 = images2 @ transform.T
# 还原交换维度，再恢复原始图像
images4 = images3.transpose(1, 2)
images5 = images4.view(*images.size())
print(f"images.shape{images.shape}")
print(f"images1.shape{images1.shape}")
print(f"images2.shape{images2.shape}")
print(f"images3.shape{images3.shape}")
print(f"images4.shape{images4.shape}")
print(f"images5.shape{images5.shape}")

#### 改用einops

In [ ]:
images = torch.randn(64, 32, 32, 3)
H = W = 32
transform = torch.randn(32 * 32, 32 * 32)
# 展平宽和高，然后交换维度
images1 = rearrange(images, "B H W C -> B C (H W)")
# 应用线形变换
images2 = einsum(images1, transform, "B C t_in, t_out t_in -> B C t_out")
# 还原交换维度，再恢复原始图像
images3 = rearrange(images2, "B C (H W) -> B H W C", H=H, W=W)
print(f"images.shape{images.shape}")
print(f"images1.shape{images1.shape}")
print(f"images2.shape{images2.shape}")
print(f"images3.shape{images3.shape}")

### 再用einx.dot

In [ ]:
H = W = 32
images = torch.randn(64, 32, 32, 3)
transform = torch.randn(32 * 32, 32 * 32)
images1 = einx.dot(
    "B H_in W_in C, (H_out W_out) (H_in W_in) -> B H_out W_out C",
    images,
    transform,
    H_out=H,
    W_out=W
)
print(f"images.shape{images.shape}")
print(f"images1.shape{images1.shape}")

### 3.3.3 嵌入

In [ ]:
from torch import nn

embedding = nn.Embedding(num_embeddings=4, embedding_dim=2)
print(embedding.weight)

token_ids = torch.tensor([[2, 3], [3, 0]])
print(token_ids.shape, len(token_ids))
embeded = embedding(token_ids)
print(embeded)